In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, roc_auc_score

RANDOM_STATE = 42
TARGET_COL = "Class"
TEST_SIZE = 0.2

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Ensure the directory exists
DATA_RAW.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Check if the file exists before reading
csv_file = DATA_RAW / "creditcard.csv"
if csv_file.exists():
    df = pd.read_csv(csv_file, low_memory=False)  # Suppress DtypeWarning
    df.head(10)
else:
    print(f"File not found: {csv_file}")


In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

In [ ]:
train_idx = np.load(SPLITS_DIR / "X_train.npy")

X_train = X.loc[train_idx].copy()
y_train = y.loc[train_idx].copy()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV

v_columns = [f"V{i}" for i in range(1, 29)]
amount_col = ["Amount"]
time_col = ["Time"]

SECONDS_IN_DAY = 86400

def time_cyc_features(X):
    t = (X % SECONDS_IN_DAY) / SECONDS_IN_DAY
    return np.hstack([np.sin(2 * np.pi * t), np.cos(2 * np.pi * t)])

def coerce_numeric_df(X):
    X = X.copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    return X

to_numeric = FunctionTransformer(coerce_numeric_df, validate=False)

amount_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, validate=True)),
    ("scaler", StandardScaler()),
])

v_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

time_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("cyc", FunctionTransformer(time_cyc_features, validate=True)),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("amount", amount_transformer, amount_col),
        ("time", time_transformer, time_col),
        ("v", v_transformer, v_columns),
    ],
    remainder="drop"
)

USE_SAMPLE_WEIGHT = True  # set True if you want it
W_POS = 200               # try 50, 100, 200

# Define the pipeline
model = Pipeline(steps=[
    ("to_numeric", to_numeric),
    ("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(
        random_state=RANDOM_STATE,
        max_iter=288,
        learning_rate=0.19063571821788408,
        max_depth=10,
        max_bins=206,
        min_samples_leaf=11,
        max_leaf_nodes=35,
        l2_regularization=0.3745401188473625,
    ))
])




In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_proba = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

calibrated_model = CalibratedClassifierCV(
    estimator=model,
    method="isotonic",
    cv=cv
)

USE_SAMPLE_WEIGHT = True

if USE_SAMPLE_WEIGHT:
    sample_weight = np.where(y_train == 1, W_POS, 1)
    calibrated_model.fit(X_train, y_train, classifier__sample_weight=sample_weight)
else:
    calibrated_model.fit(X_train, y_train)

## Why OOF Calibrated Probabilities?

For threshold selection, we must use **out-of-fold (OOF) probabilities** rather than in-sample probabilities to avoid optimistic/biased metrics.

Using `calibrated_model.predict_proba(X_train)` would give in-sample probabilities on the same data used for calibration, leading to overly optimistic threshold selection.

Instead, we use `cross_val_predict` with a `CalibratedClassifierCV` estimator to obtain honest OOF calibrated probabilities.

In [ ]:
# Use cross_val_predict with CalibratedClassifierCV to get OOF calibrated probabilities
# This ensures honest threshold selection (not optimistic/in-sample)
from sklearn.calibration import CalibratedClassifierCV
calibrated_for_oof = CalibratedClassifierCV(estimator=model, method="isotonic", cv=cv)
oof_proba = cross_val_predict(calibrated_for_oof, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

# Sanity checks
assert len(oof_proba) == len(y_train), "oof_proba length != y_train length"
if np.isnan(oof_proba).any():
    raise ValueError("oof_proba contains NaNs")
print("oof_proba shape:", oof_proba.shape)

precision, recall, thresholds = precision_recall_curve(y_train, oof_proba)
thr_ext = np.r_[0.0, thresholds]
print(len(precision), len(recall), len(thr_ext))

In [ ]:
import sys
from pathlib import Path

# Add the parent directory of 'src' to sys.path
sys.path.append(str(PROJECT_ROOT))

from src.choose_thresholds import choose_threshold_max_recall_at_fpr

chosen_thr, strategy, metrics = choose_threshold_max_recall_at_fpr(
    y_true=y_train,
    proba=oof_proba,
    max_fpr=0.001
)
print(f"Chosen threshold: {chosen_thr}, strategy: {strategy}")
print("Metrics at chosen threshold:")
for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {round(metric_value, 4)}")

In [ ]:
cm = confusion_matrix(y_train, (oof_proba >= chosen_thr).astype(int))
print("Confusion Matrix:\n", cm)

In [ ]:
#PR curve for chosen threshold

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label="Precision-Recall Curve")
plt.scatter(metrics['recall'], metrics['precision'], color='red', label='Chosen Threshold', zorder=5)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve with Chosen Threshold (HGB)")
plt.legend()
plt.grid()
plt.savefig(REPORTS_DIR / "pr_curve_chosen_threshold_hgb.png")
plt.show()

#Recall@FixedPRecision graph

plt.figure(figsize=(8, 6))
plt.plot(thr_ext, recall, label="Recall vs Threshold")
plt.scatter(chosen_thr, metrics['recall'], color='red', label='Chosen Threshold', zorder=5)
plt.xlabel("Threshold")
plt.ylabel("Recall")
plt.title("Recall vs Threshold with Chosen Threshold (HGB)")
plt.legend()
plt.grid()
plt.savefig(REPORTS_DIR / "recall_vs_threshold_chosen_threshold_hgb.png")



In [ ]:
import numpy as np
from scipy.stats import uniform, randint

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_curve
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold

# --- 1) Param search space (HGB) ---
param_distributions = {
    "classifier__learning_rate": uniform(0.01, 0.19),
    "classifier__max_leaf_nodes": randint(15, 50),
    "classifier__min_samples_leaf": randint(5, 30),
    "classifier__max_iter": randint(100, 600),
}

# --- 2) Custom scorer: maximize recall (TPR) subject to FPR <= max_fpr ---
def recall_at_max_fpr(y_true, y_score, max_fpr=0.001, **kwargs):
    """
    y_score is expected to be probability of positive class (fraud).
    **kwargs is required because newer sklearn may pass extra kwargs (e.g., needs_proba).
    """
    fpr, tpr, _ = roc_curve(y_true, y_score)
    ok = np.where(fpr <= max_fpr)[0]
    return float(np.max(tpr[ok])) if ok.size else 0.0

recall_fpr_0p1_scorer = make_scorer(
    recall_at_max_fpr,
    needs_proba=True,   # tells sklearn to pass predict_proba output
    max_fpr=0.001       # passed to recall_at_max_fpr as kwarg
)

# --- 3) Multi-metric scoring; refit by your KPI ---
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "recall_fpr_0p1": recall_fpr_0p1_scorer,
}

# If you haven't defined skf yet, define it like this:
# (shuffle is important for randomized search reproducibility)
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=40,
    scoring=scoring,
    refit="recall_fpr_0p1",
    n_jobs=-1,
    cv=cv,
    random_state=RANDOM_STATE,
    verbose=2,
    error_score="raise",  # fail fast if something else breaks
)

# --- 4) class imbalance handling for HGB via sample_weight ---
# If you want "balanced" behavior for HistGradientBoostingClassifier, use sample weights:
USE_SAMPLE_WEIGHT = True  # set True if you want it
W_POS = 200              # try 50, 100, 200

if USE_SAMPLE_WEIGHT:
    sample_weight = np.where(y_train == 1, W_POS, 1.0)
    random_search.fit(X_train, y_train, classifier__sample_weight=sample_weight)
else:
    random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)
print("Best Score (Recall at FPR 0.1%):", random_search.best_score_)


In [ ]:
# Set the best parameters for the model

best_model = random_search.best_estimator_


# Refit best_model on full training set for final model serving
# (OOF probabilities below are still used for threshold tuning)
best_model.fit(X_train, y_train)

oof_proba = cross_val_predict(best_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
chosen_thr, strategy, metrics = choose_threshold_max_recall_at_fpr(
    y_true=y_train,
    proba=oof_proba,
    max_fpr=0.001
)

print(f"Chosen threshold after hyperparameter tuning: {chosen_thr}, strategy: {strategy}")
print("Metrics at chosen threshold after hyperparameter tuning:")
for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {round(metric_value, 4)}")

